# Neural Prototyping

# Google Colab Mounting

In [1]:
!rm -rf /content/credit-risk-modeling
!git clone https://github.com/BillyBrothers/credit-risk-modeling.git
!pip install -r /content/credit-risk-modeling/requirements.txt

import sys 
sys.path.append('/content/credit-risk-modeling')

Cloning into 'credit-risk-modeling'...
remote: Enumerating objects: 1181, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 1181 (delta 38), reused 37 (delta 17), pack-reused 1121 (from 1)
Receiving objects: 100% (1181/1181), 52.27 MiB | 19.50 MiB/s, done.
Resolving deltas: 100% (800/800), done.
Updating files: 100% (68/68), done.


In [2]:
!pip install scikeras

In [3]:
pip install -q -U keras-tuner

In [4]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno

# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay
from credit_risk_modeling import model_eval

import tensorflow as tf
from tensorflow import keras
from keras import layers
from scikeras.wrappers import KerasClassifier
import keras_tuner as kt

2026-02-05 18:05:43.689 | INFO     | credit_risk_modeling.config:<module>:11 - PROJ_ROOT path is: /content/credit-risk-modeling


In [5]:
!ls

credit-risk-modeling  logs  sample_data  untitled_project


In [6]:
!ls credit-risk-modeling

credit_risk_modeling  LICENSE	 pyproject.toml  requirements.txt
data		      Makefile	 README.md	 tests
docs		      models	 references
environment.yml       notebooks  reports


# Imports

In [7]:
X_train = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_train_neural.csv"
)

In [8]:
X_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_test_neural.csv"
)

In [9]:
X_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_val_neural.csv"
)

In [10]:
y_train= pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_train.csv"
)
y_train = y_train.values.ravel()
neg, pos = np.bincount(y_train)
total = neg + pos
print(f"Examples:\n The training sets total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The training sets total amount of samples: 22686
 Positive: 4962 (21.87% of total)


In [11]:
y_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_test.csv"
)
y_test = y_test.values.ravel()
neg, pos = np.bincount(y_test)
total = neg + pos
print(f"Examples:\n The testing set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The testing set total amount of samples: 2917
 Positive: 638 (21.87% of total)


In [12]:
y_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_val.csv"
)
y_val = y_val.values.ravel()
neg, pos = np.bincount(y_val)
total = neg + pos
print(f"Examples:\n The validation set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The validation set total amount of samples: 6806
 Positive: 1488 (21.86% of total)


# Build Sequential Models

Includes hyperparameter tuning

In [13]:
mlp_models = []
mlp_results = []

In [ ]:
# def mlp_model1(model):
#     model = keras.Sequential(name='MLP-1')
#     model.add(keras.Input(shape=(X_train.shape[1], ))),
#     model.add(layers.Dense(units=64, activation='relu')),
#     model.add(layers.Dropout(rate= 0.20)),
#     model.add(layers.Dense(units=1,activation='sigmoid')),
#     model.compile(
#         optimizer='adam',
#         loss= keras.losses.BinaryCrossentropy(),
#         metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
# )

SyntaxError: unmatched ')' (ipython-input-2152272452.py, line 11)

In [15]:
# Architecture 1: Single hidden dense layer
def mlp1(hp):
    model1 = keras.Sequential(name='MLP-1')
    model1.add(keras.Input(shape=(X_train.shape[1], ))),
    hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
    model1.add(layers.Dense(units=hp_units, activation='relu')),
    model1.add(layers.Dropout(rate= 0.20)),
    model1.add(layers.Dense(units=1,activation='sigmoid')),
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model1.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss=keras.losses.BinaryCrossentropy(),
                metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
    ) 
    return model1

In [ ]:
# def mlp_model2(model):
#     model = keras.Sequential(name='MLP-2')
#     model.add(keras.Input(shape=(X_train.shape[1], ))),
#     model.add(layers.Dense(units=64, activation='relu')),
#     model.add(layers.Dropout(rate= 0.20)),
#     model.add(layers.Dense(units=128, activation='relu')),
#     model.add(layers.Dropout(rate=0.20)),
#     model.add(layers.Dense(units=1,activation='sigmoid')),
#     model.compile(
#         optimizer='adam',
#         loss= keras.losses.BinaryCrossentropy(),
#         metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
# )

In [16]:
# Architecture 2: Two hidden dense layers
def mlp2(hp):
    model2 = keras.Sequential(name='MLP-2')
    model2.add(keras.Input(shape=(X_train.shape[1], ))),
    hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
    model2.add(layers.Dense(units=hp_units, activation='relu')),
    model2.add(layers.Dropout(rate= 0.20)),
    hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
    model2.add(layers.Dense(units=hp_units, activation='relu')),
    model2.add(layers.Dropout(rate=0.20)),
    model2.add(layers.Dense(units=1,activation='sigmoid')),
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model2.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss=keras.losses.BinaryCrossentropy(),
                metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
    )
    return model2

In [ ]:
# def mlp_model3(model):
#     model = keras.Sequential(name='MLP-3')
#     model.add(keras.Input(shape=(X_train.shape[1], ))),
#     model.add(layers.Dense(units=64, activation='relu')),
#     model.add(layers.Dropout(rate= 0.20)),
#     model.add(layers.Dense(units=128, activation='relu')),
#     model.add(layers.Dropout(rate=0.20)),
#     model.add(layers.Dense(units=256, activation='relu')),
#     model.add(layers.Dropout(rate=0.2)),
#     model.add(layers.Dense(units=1,activation='sigmoid')),
#     model.compile(
#         optimizer='adam',
#         loss= keras.losses.BinaryCrossentropy(),
#         metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
# )

In [18]:
# Architecture 3: Three hidden dense layers
def mlp3(hp):
    model3 = keras.Sequential(name='MLP-3')
    model3.add(keras.Input(shape=(X_train.shape[1], ))),
    hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
    model3.add(layers.Dense(units=hp_units, activation='relu')),
    model3.add(layers.Dropout(rate= 0.20)),
    hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
    model3.add(layers.Dense(units=hp_units, activation='relu')),
    model3.add(layers.Dropout(rate=0.20)),
    hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
    model3.add(layers.Dense(units=hp_units, activation='relu')),
    model3.add(layers.Dropout(rate=0.2)),
    model3.add(layers.Dense(units=1,activation='sigmoid')),
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model3.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss=keras.losses.BinaryCrossentropy(),
                metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
    )

    return model3

### Callbacks

In [23]:
reduce_lr_plateau = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    verbose=1,
    min_lr=0.001
)

In [24]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=7,
    verbose=1,
    restore_best_weights=True
)

In [25]:
log_dir = "logs/fit/"
tensorboard = keras.callbacks.TensorBoard(
    log_dir= log_dir
)

### Hyperparameter Tuning

In [26]:
mlp_models = [
    ("MLP-1", mlp1),
    ("MLP-2", mlp2),
    ("MLP-3", mlp3)
]

In [27]:
tuner = kt.Hyperband(
    hypermodel= mlp1,
    objective= "val_AUC",
    max_epochs= 30,
    factor=3,
)

Reloading Tuner from ./untitled_project/tuner0.json


In [29]:
tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_data= (X_val, y_val),
    callbacks = [early_stopping]
)

### Compile

In [ ]:
for model_name, model in mlp_models:
    model.compile(
        optimizer='adam',
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)

In [ ]:
class_weight = {
    0: 1.0,
    1: 2.0
    }

### Fit

In [ ]:
for model_name, model in mlp_models:
    print(f"Currently fitting model {model_name}.")
    history = model.fit(
        x= X_train,
        y= y_train,
        batch_size=32,
        epochs= 100,
        verbose=2,
        callbacks= [early_stopping, reduce_lr_plateau, tensorboard],
        validation_data= (X_val, y_val),
        class_weight= class_weight
    )

Currently fitting model MLP-64.
Epoch 1/100
709/709 - 4s - 5ms/step - auc: 0.8483 - loss: 0.5564 - val_auc: 0.8884 - val_loss: 0.3547 - learning_rate: 1.0000e-03
Epoch 2/100
709/709 - 2s - 3ms/step - auc: 0.8872 - loss: 0.4846 - val_auc: 0.9003 - val_loss: 0.3267 - learning_rate: 1.0000e-03
Epoch 3/100
709/709 - 2s - 3ms/step - auc: 0.8954 - loss: 0.4645 - val_auc: 0.9043 - val_loss: 0.3160 - learning_rate: 1.0000e-03
Epoch 4/100
709/709 - 2s - 3ms/step - auc: 0.8983 - loss: 0.4563 - val_auc: 0.9061 - val_loss: 0.3085 - learning_rate: 1.0000e-03
Epoch 5/100
709/709 - 3s - 4ms/step - auc: 0.9012 - loss: 0.4475 - val_auc: 0.9076 - val_loss: 0.3164 - learning_rate: 1.0000e-03
Epoch 6/100
709/709 - 2s - 3ms/step - auc: 0.9042 - loss: 0.4390 - val_auc: 0.9089 - val_loss: 0.3084 - learning_rate: 1.0000e-03
Epoch 7/100
709/709 - 2s - 3ms/step - auc: 0.9027 - loss: 0.4395 - val_auc: 0.9103 - val_loss: 0.2989 - learning_rate: 1.0000e-03
Epoch 8/100
709/709 - 2s - 3ms/step - auc: 0.9066 - loss: 

### Inference

In [ ]:
y_pred = model1.predict(
    X_test,
    verbose=1
)

92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
models_only = []

In [ ]:
for model_name, model in mlp_models:
    model = KerasClassifier(model)
    models_only.append(model)

In [ ]:
models_only

[KerasClassifier(
 	model=<Sequential name=MLP-64, built=True>
 	build_fn=None
 	warm_start=False
 	random_state=None
 	optimizer=rmsprop
 	loss=None
 	metrics=None
 	batch_size=None
 	validation_batch_size=None
 	verbose=1
 	callbacks=None
 	validation_split=0.0
 	shuffle=True
 	run_eagerly=False
 	epochs=1
 	class_weight=None
 ),
 KerasClassifier(
 	model=<Sequential name=MLP-64-128, built=True>
 	build_fn=None
 	warm_start=False
 	random_state=None
 	optimizer=rmsprop
 	loss=None
 	metrics=None
 	batch_size=None
 	validation_batch_size=None
 	verbose=1
 	callbacks=None
 	validation_split=0.0
 	shuffle=True
 	run_eagerly=False
 	epochs=1
 	class_weight=None
 ),
 KerasClassifier(
 	model=<Sequential name=MLP-64-128-256, built=True>
 	build_fn=None
 	warm_start=False
 	random_state=None
 	optimizer=rmsprop
 	loss=None
 	metrics=None
 	batch_size=None
 	validation_batch_size=None
 	verbose=1
 	callbacks=None
 	validation_split=0.0
 	shuffle=True
 	run_eagerly=False
 	epochs=1
 	class_we

In [ ]:
untuned_model_performance, fitted_models = model_eval.comparing_models(
    models= models_only,
    X_train= X_train,
    y_train= y_train,
    X_test= X_test,
    y_test= y_test
)

709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - auc: 0.9218 - loss: 0.2466
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - auc: 0.9217 - loss: 0.2349
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
709/709 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - auc: 0.9211 - loss: 0.2367
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


## Hyperparameter Tuning